# RAG Workflow for ESG Sustainability Report Analysis

**Task:** Retrieve and answer ESG questions from four steel company sustainability PDFs using a RAG pipeline,
comparing output quality and latency across four open-source LLMs.

**Companies covered:**
- JFE Holdings (Japan) — Sustainability Report 2025
- JSW Steel (India) — Business Responsibility & Sustainability Report FY2024-25
- POSCO (South Korea) — Sustainability Report 2024
- Tata Steel UK — Sustainability Report 2023-2025

---

## Architecture Overview

```
PDF Files
   │
   ▼
[PDF Loader (pdfplumber)] — page-level text extraction
   │
   ▼
[Chunker] — table-aware: tables as atomic chunks + sliding window (512 chars, 64 overlap) on remaining text
   │
   ▼
[Embedder] — sentence-transformers (all-MiniLM-L6-v2)
   │
   ▼
[FAISS Vector Store] — cosine-similarity retrieval (IndexFlatIP)
   │
   ▼
[4x LLMs via vLLM] — sequential load/run/unload per model
   │
   ▼
[Output] — per-question blocks + side-by-side table + CSV/JSON export
```

### Key Design Choices

| Decision | Choice | Rationale |
|---|---|---|
| PDF parser | `pdfplumber` | Spatially-aware extraction; handles multi-column tables better than `pypdf` |
| Chunk size | 512 chars, 64-char overlap | Robust to noisy PDF text; overlap prevents numbers being split across boundaries |
| Chunk boundary | Within-page only | Guarantees exact page citations — critical for auditable ESG data |
| Embedder | `all-MiniLM-L6-v2` | Fast, CPU-friendly, strong zero-shot retrieval; shared across all models for fair comparison |
| Vector store | `FAISS IndexFlatIP` | Exact search; justified by corpus size (3,379 vectors, 5.2MB) |
| LLMs | 4x open-source via vLLM | Multi-model comparison; sequential load/unload fits single H100 80GB |
| Precision | fp16 (bfloat16 for Gemma) | Same effective precision across all models — fair comparison |
| Temperature | 0.1 | Near-deterministic output; reduces hallucination on numerical ESG data |
| Top-k retrieval | 8 chunks | Balances recall vs context window; ESG numbers often span multiple tables |


## 1. Install Dependencies

In [1]:
# ! pip install bitsandbytes>=0.46.1
# ! pip install rank-bm25

In [2]:
# ! python -m venv ddl_venv
# ! source ddl_venv/bin/activate

In [3]:
# Run once
# !pip install pdfplumber sentence-transformers faiss-cpu numpy pandas python-dotenv vllm -q


In [4]:
# ! python -m ipykernel install --user --name=ddl_venv

## 2. Imports

All configuration, dataclasses, and helper functions live in `rag_classes.py`.
Ground truth tables live in `ground_truth.py`.

In [5]:
import pandas as pd

from Config       import PDF_FILES, PDF_DIR, CHUNK_DIR, TOP_K_PER_COMPANY, MODEL_MAP
from DataClasses import Chunk
from Chunking     import load_and_chunk, save_chunks
from VectorStore import VectorStore
from RAGEngine   import format_context, rag_query_single_model, run_all_models
from GroundTruth import get_dataframes, save_all as save_ground_truth

from Evaluator   import evaluate_all, print_report

print('Imports OK.')

Imports OK.


## 3. PDF Loading & Chunking

PDF loading and chunking are done in a single pass while the file handle is open —
required because `pdfplumber`'s `extract_tables()` needs the live page object.
Pages with no extractable content are skipped.

In [ ]:
chunks = load_and_chunk(PDF_FILES, PDF_DIR)

## 4. Chunking

Table-aware chunker: tables on each page are extracted as atomic chunks (never split).
Remaining text is split with a sliding window (512 chars, 64 overlap), within-page only.
Chunks saved to `chunks/` as one `.txt` file per company for inspection.

In [ ]:
save_chunks(chunks, CHUNK_DIR)

In [ ]:
# Quick sanity check
df_meta = pd.DataFrame([
    {"company": c.company, "page": c.page}
    for c in chunks
])
print("\nChunks per company:")
print(df_meta.groupby("company").size().to_string())

## 5. Embedding & FAISS Vector Index

Builds a full index and a per-company sub-index.
Hybrid retrieval: FAISS top-10 + BM25 top-10 per company → up to 80 candidates → cross-encoder reranked → top 40 passed to LLM.

In [ ]:
vs = VectorStore()
vs.build(chunks)

In [ ]:
# After building
print(f"Chunks: {vs.index.ntotal}")
print(f"Vector dimension: {vs.index.d}")
print(f"Approximate memory: {vs.index.ntotal * vs.index.d * 4 / 1e6:.1f} MB")

## 6. RAG Query Engine

Models are loaded sequentially (one at a time) to fit on a single H100 80GB.
See `rag_classes.py` for `rag_query_single_model` and `run_all_models`.

## 7. Define Questions & Run All Models

Three questions are run against all four models.
Results are collected into a single DataFrame for comparison.


In [ ]:
# QUESTIONS = {
#     "Q1_emissions_energy": (
#         "Extract Scope 1, Scope 2, and Scope 3 GHG emissions (in million tonnes CO2e) "
#         "and total energy consumption for calendar/fiscal years 2023, 2024, and 2025 "
#         "for each of the four companies: JFE Holdings, JSW Steel, POSCO, and Tata Steel UK. "
#         "If data is not present for a year, leave blank. Include units for all values."
#     ),
#     "Q2_ccus": (
#         "Which of the four companies (JFE Holdings, JSW Steel, POSCO, Tata Steel UK) "
#         "have installed, operated, piloted, or tested any carbon capture, utilisation "
#         "or storage (CCUS, CCU, CCS) technology? For each, describe the technology, "
#         "current status (planned / piloted / operational), scale, and year."
#     ),
#     "Q3_net_zero": (
#         "What are the net-zero or carbon neutrality targets for each of the four companies: "
#         "JFE Holdings, JSW Steel, POSCO, and Tata Steel UK? "
#         "Include the target year, scope of the commitment (Scope 1/2/3), "
#         "any interim 2030 targets, and whether the target is validated by SBTi or another body."
#     ),
# }


QUESTIONS = {
    "Q1_emissions_energy": "Extract Scope 1, Scope 2, Scope 3 emissions and total energy consumption for 2023, 2024 and 2025 for all companies. If the data is not present, leave the result empty.",
    "Q2_ccus": "Which companies have installed, operated or tested carbon capture, utilisation or storage (CCUS) technology?", 
    "Q3_net_zero": "What is the net-zero target of each company?"
}

# Run all models sequentially
# This will take ~10-20 min depending on GPU and model sizes
df_results = run_all_models(QUESTIONS, vs)                          # → rag_run.log

print(f"\nTotal results: {len(df_results)} rows ({len(MODEL_MAP)} models x {len(QUESTIONS)} questions)")
print(df_results[["model", "model_size", "quantization", "question_label", "latency_seconds", "n_tokens", "finish_reason"]].to_string(index=False))


In [ ]:
# ── Automated evaluation against ground truth ──────────────────────────
scores = evaluate_all(df_results)
print_report(scores)

scores.to_csv("evaluation_scores.csv", index=False)
print("Saved evaluation_scores.csv")

## 8. Output 1 — Per-Question Answer Blocks

Print all four model answers under each question for direct reading.


In [ ]:
for q_label in QUESTIONS:
    print("\n" + "=" * 70)
    print(f"QUESTION: {q_label}")
    print("=" * 70)
    print(f"\n{QUESTIONS[q_label]}\n")

    subset = df_results[df_results["question_label"] == q_label]
    for _, row in subset.iterrows():
        print(f"\n--- {row['model'].upper()} ({row['model_id']}) "
              f"| {row['latency_seconds']}s | {row['n_tokens']} tokens ---")
        print(row["answer"])
        print()


In [ ]:
df_pivot = df_results.pivot(
    index="question_label",
    columns="model",
    values="answer"
).reset_index()

# Also pivot latency for comparison
df_latency = df_results.pivot(
    index="question_label",
    columns="model",
    values="latency_seconds"
).reset_index()
df_latency.columns = ["question_label"] + [f"{m}_latency_s" for m in MODEL_MAP]

print("Side-by-side answers (truncated to 200 chars per cell for display):")
display_pivot = df_pivot.copy()
for col in MODEL_MAP:
    if col in display_pivot.columns:
        display_pivot[col] = display_pivot[col].str[:200] + "..."

print(display_pivot.to_string(index=False))
print("\nLatency comparison (seconds):")
print(df_latency.to_string(index=False))


## 9. Output 3 — Export to CSV and JSON


In [ ]:
import csv

# Full results — one row per (model, question)
df_results.to_csv("model_comparison_results.csv", index=False, quoting=csv.QUOTE_ALL)
df_results.to_json("model_comparison_results.json", orient="records", indent=2)
print("Saved model_comparison_results.csv / .json")

# Side-by-side pivot
df_pivot.to_csv("model_comparison_pivot.csv", index=False, quoting=csv.QUOTE_ALL)
print("Saved model_comparison_pivot.csv")

# Latency summary
df_latency.to_csv("model_latency.csv", index=False)
print("Saved model_latency.csv")

print("\nAll outputs saved.")


## 10. Bonus — Agentic Decision Logic

Below is a lightweight agent router that decides **which tool to use** for a given query without calling an LLM for simple classification. This demonstrates the principle requested in the bonus section.

### Agent Strategy

| Signal in query | Decision | Reasoning |
|---|---|---|
| Mentions a company name + data type already in our PDF corpus | `retrieve_from_pdfs` | Closed-domain; PDF is ground truth |
| Asks about recent events / current prices / live data | `web_search` | Post-cutoff; PDF is stale |
| Question is ambiguous (no entity, no time frame) | `request_clarification` | Retrieval would be noisy |
| Asks for a structured table with specific column schema | `structured_extraction` | Triggers JSON-mode prompt |
| Entity in query not in our PDF set | `web_search` | Out-of-corpus |

In [ ]:
KNOWN_COMPANIES = {"jfe", "jsw", "posco", "tata steel"}
LIVE_DATA_SIGNALS = ["current price", "today", "latest news", "stock", "2026"]
AMBIGUITY_SIGNALS = ["some company", "a steel company", "they", "it"]
STRUCTURED_SIGNALS = ["table", "csv", "json", "extract", "structured", "list all"]


def route_query(query: str):
    """
    Lightweight rule-based agent router.
    
    Returns a dict: {action, reason, params}
    In a full agentic system this would be an LLM tool-selection call;
    here we use heuristics to demonstrate the logic cheaply and deterministically.
    """
    q_lower = query.lower()

    # Rule 1: ambiguity check — ask for clarification
    if any(sig in q_lower for sig in AMBIGUITY_SIGNALS):
        return {
            "action": "request_clarification",
            "reason": "Query is ambiguous — no specific company identified.",
            "params": {"clarification_needed": "Which company are you referring to?"},
        }

    # Rule 2: live/web data
    if any(sig in q_lower for sig in LIVE_DATA_SIGNALS):
        return {
            "action": "web_search",
            "reason": "Query requires real-time or post-publication data.",
            "params": {"search_query": query},
        }

    # Rule 3: company not in our corpus
    company_in_corpus = any(co in q_lower for co in KNOWN_COMPANIES)
    mentions_a_company = any(w[0].isupper() for w in query.split())
    if mentions_a_company and not company_in_corpus:
        return {
            "action": "web_search",
            "reason": "Named entity not found in our PDF corpus.",
            "params": {"search_query": query},
        }

    # Rule 4: structured extraction
    if any(sig in q_lower for sig in STRUCTURED_SIGNALS):
        return {
            "action": "structured_extraction",
            "reason": "User wants structured/tabular output.",
            "params": {"output_format": "json", "query": query},
        }

    # Default: retrieve from PDFs
    return {
        "action": "retrieve_from_pdfs",
        "reason": "Query is within our PDF corpus scope.",
        "params": {"query": query, "top_k_per_company": TOP_K_PER_COMPANY},
    }


# Demo
test_queries = [
    "What is POSCO's net-zero target?",
    "What is ArcelorMittal's carbon footprint today?",
    "Give me a table of all Scope 1 emissions",
    "Some company mentioned CCUS — what did they say?",
]

print("Agent routing demo:")
for q in test_queries:
    decision = route_query(q)
    print(f"\nQuery : {q}")
    print(f"Action: {decision['action']}")
    print(f"Reason: {decision['reason']}")

## 12. Ground Truth

Pre-extracted reference answers (line-by-line from PDFs) live in `ground_truth.py`.
Run `python ground_truth.py` to print all tables and save CSV/JSON.
Use `get_dataframes()` here to load them for comparison against RAG output.

In [ ]:
df_gt_emissions, df_gt_ccus, df_gt_nz = get_dataframes()

print('Q1 — Emissions & Energy')
print(df_gt_emissions.to_string(index=False))

print('\nQ2 — CCUS')
print(df_gt_ccus[['company','technology','status','source_pages']].to_string(index=False))

print('\nQ3 — Net-Zero')
print(df_gt_nz[['company','net_zero_year','commitment_label','interim_2030_target']].to_string(index=False))

# Optionally save
# save_ground_truth()

---

## 13. Approach Notes & Limitations

### What works well
- **Page-level citations** are preserved end-to-end from ingestion through retrieval to the final answer.
- **Dual output mode** (plain text + JSON) lets downstream systems parse results programmatically.
- **FAISS flat index** gives exact nearest-neighbour results — no recall loss from approximation.
- Pre-computed ground truth tables provide a deterministic fallback and enable evaluation of RAG accuracy.

### Limitations & mitigations

| Limitation | Impact | Mitigation |
|---|---|---|
| **POSCO report is partially in Korean** | Embedding model may retrieve less relevant Korean chunks | Used English-heavy pages for POSCO; a multilingual model (e.g., `paraphrase-multilingual-MiniLM-L12-v2`) would improve this |
| **JFE/POSCO PDFs have doubled-character artifacts** (e.g., `SSuussttaaiinnaabbiilliittyy`) | Noisy embeddings for nav-bar text | Artefacts appear on navigation headers, not data tables; minimal impact on numerical retrieval |
| **Numeric OCR ambiguity** (Indian number system in JSW) | 5,31,00,751 ≠ 531,007,51 | Handled by the LLM prompt instruction; ground-truth table provides manual verification |
| **Different fiscal year conventions** across companies | 2023 means different periods per company | Reported year is preserved alongside calendar year in the ground-truth table |
| **JFE does not disclose Scope 3** in main ESG data table | Scope 3 cell is null for JFE | Noted explicitly; partial Scope 3 data may be in subsidiary sections |
| **No persistent vector store** | Index rebuilt each run (~2 min) | For production, serialise with `faiss.write_index()` and a metadata pickle |
| **Hybrid retrieval overhead** | BM25 + cross-encoder add ~15s per query | Cross-encoder runs on CPU; acceptable since GPU is reserved for the LLM |

### Scaling to production
```python
# Persist index
faiss.write_index(vs.index, "esg_index.faiss")
import pickle
with open("esg_chunks.pkl", "wb") as f:
    pickle.dump(vs.chunks, f)

# Reload
vs.index = faiss.read_index("esg_index.faiss")
with open("esg_chunks.pkl", "rb") as f:
    vs.chunks = pickle.load(f)
```